# Automated Code Review with LangGraph
### Notebook 1 of 3 — Foundations: State, Nodes & Graph

This notebook covers the core building blocks of the agentic code review tool.

| Notebook | Focus |
|----------|-------|
| **01 — Foundations** (this one) | State, nodes, graph build, graph visualisation |
| 02 — Demo & Patterns | Run all 4 paths, streaming, persistence, token cost |
| 03 — Advanced Patterns | Unit testing, LLM-as-Judge, extensions |

**By the end of this notebook** you will have built and compiled a working 12-node LangGraph
that routes pull requests through parallel security + quality review agents.

**Prerequisites:** OpenAI API key in `.env` file (`OPENAI_API_KEY=sk-...`)

## Architecture Overview

The complete system (v2, including LLM-as-Judge) consists of **12 nodes** connected by conditional and unconditional edges across three distinct execution paths.

```mermaid
flowchart TD
    START([__start__]) --> read_pr

    read_pr["📄 read_pr\nReads PR diff from disk"]
    read_pr --> route_review

    route_review{"🔀 route_review\nLLM classifies diff\nsimple / full / error"}
    route_review -->|simple| simple_review
    route_review -->|full| full_review_start
    route_review -->|error| return_final_answer

    simple_review["✏️ simple_review\nLightweight LLM review\nStructured output"]
    simple_review --> final_decision

    full_review_start["⚡ full_review_start\nNo-op fan-out trigger"]
    full_review_start --> security_review
    full_review_start --> quality_analysis

    security_review["🔐 security_review\nSecurity Engineer agent\nFinds BLOCKING issues"]
    quality_analysis["👨‍💻 quality_analysis\nSenior Developer agent\nStyle + bug analysis"]

    quality_analysis --> quality_judge

    quality_judge{"⚖️ quality_judge\ngpt-4o-mini scores\nanalysis 1-10"}
    quality_judge -->|score >= 7| aggregate
    quality_judge -->|score < 7, retries left| quality_analysis

    security_review --> aggregate

    aggregate["🔗 aggregate\nFan-in join point\nWaits for both agents"]
    aggregate --> summarize_findings

    summarize_findings["👔 summarize_findings\nTech Lead synthesis\nStructured output with fix list"]
    summarize_findings -->|BLOCKING| blocked_review
    summarize_findings -->|NON-BLOCKING| final_decision

    blocked_review["🚫 blocked_review\nLogs hard stop\nSummary JSON preserved"]
    blocked_review --> final_decision

    final_decision["🏁 final_decision\nAPPROVED / REQUEST CHANGES / REJECT"]
    final_decision --> return_final_answer

    return_final_answer["📢 return_final_answer\nTerminal display node\nHandles errors too"]
    return_final_answer --> END([__end__])

    style START fill:#e8f5e9,stroke:#388e3c
    style END fill:#e8f5e9,stroke:#388e3c
    style route_review fill:#fff9c4,stroke:#f9a825
    style quality_judge fill:#fff9c4,stroke:#f9a825
    style aggregate fill:#fff9c4,stroke:#f9a825
    style blocked_review fill:#ffebee,stroke:#c62828
    style return_final_answer fill:#e3f2fd,stroke:#1565c0
    style full_review_start fill:#f3e5f5,stroke:#6a1b9a
```

### Node roles at a glance

| Node | Type | Role |
|------|------|------|
| `read_pr` | I/O | Reads file, stores errors in state on failure |
| `route_review` | LLM router | Classifies diff, writes `review_type` |
| `simple_review` | LLM agent | Lightweight structured review (simple path) |
| `full_review_start` | Fan-out trigger | No-op — spawns parallel agents |
| `security_review` | LLM agent | Security Engineer (parallel) |
| `quality_analysis` | LLM agent | Senior Developer (parallel) |
| `quality_judge` | LLM-as-Judge | `gpt-4o-mini` scores analysis 1–10, triggers retry if score < 7 |
| `aggregate` | Fan-in join | No-op — synchronises parallel branches |
| `output_guardrail` | Edge function | Checks BLOCKING verdict — no state update |
| `blocked_review` | Hard stop | Logs the block — summary JSON preserved for display |
| `summarize_findings` | LLM agent | Tech Lead structured synthesis — always runs on full path |
| `final_decision` | LLM agent | Extracts APPROVED/REQUEST CHANGES/REJECT |
| `return_final_answer` | Terminal | Displays result, handles errors |

## LangGraph Concepts Primer

Before diving into the code, here are the four building blocks you need to understand.

---

### 1. State — the shared blackboard

All nodes read from and write to a single `TypedDict`. LangGraph passes the current state into every node and applies the returned dict as a **patch**:

```python
class PRReviewState(TypedDict):
    pr_content:  str                           # plain field — last write wins
    errors:      Annotated[list, operator.add] # reducer field — lists are concatenated
    tokens_used: Annotated[dict, merge_usage]  # reducer field — dicts are merged by summing
```

- **Plain fields** are overwritten on each update.
- **Reducer fields** use a merge function. This is essential for parallel branches that both write to the same field.

---

### 2. Nodes — pure Python functions

```python
def my_node(state: PRReviewState) -> dict:
    # Read from state, do work (call LLM, read file, etc.)
    return {"field_name": new_value}   # return ONLY what changed
```

- Takes the full current state as input.
- Returns a partial dict — only the fields it wants to update.
- LangGraph applies the patch (using reducers where defined).
- Nodes never need to return unchanged fields.

---

### 3. Edges — how control flows

| Edge type | API | When to use |
|-----------|-----|-------------|
| Unconditional | `add_edge(A, B)` | Always go from A to B |
| Conditional | `add_conditional_edges(A, fn, {label: node})` | Branch based on `fn(state)` return value |
| Fan-out | `add_edge(X, B)` + `add_edge(X, C)` | Run B and C in parallel |
| Fan-in | All branches point to one node | LangGraph waits for all predecessors |

Edge functions (`fn` above) are **not** nodes — they don't update state. They simply inspect state and return a label string that selects the next node.

---

### 4. Reducers — merging parallel writes

When two parallel nodes write to the same state field, a reducer decides how to combine them:

```
  security_review  ──► tokens_used = {in: 474, out: 497}
                                           ↓  merge_usage()
  quality_analysis ──► tokens_used = {in: 439, out: 631}
                                           ↓
                    final: {in: 913, out: 1128}   ← both counted
```

Without a reducer, the second write silently overwrites the first.

## Step 0 — Install Dependencies

Run this cell first. It installs all packages needed for the notebook.

In [1]:
# Install required packages
!pip install langgraph langchain-openai langchain-core python-dotenv grandalf langgraph-checkpoint-sqlite -q

## Step 1 — Install & Import

### Packages

| Package | Role |
|---------|------|
| `langgraph` | Graph execution engine — nodes, edges, state management |
| `langchain-openai` | `ChatOpenAI` wrapper for GPT-4o |
| `langchain-core` | `HumanMessage`, `SystemMessage` message types |
| `python-dotenv` | Loads `OPENAI_API_KEY` from a `.env` file |
| `grandalf` | ASCII graph rendering for `print_ascii()` |

### Key design decisions

**`load_dotenv` before `ChatOpenAI()`** — critical ordering: the API key must be in the environment *before* the LLM object is instantiated. Moving these two lines apart is a common source of `MissingCredentials` errors.

**`NOTEBOOK_DIR = Path.cwd()`** — captures the working directory at import time. Without this, relative PR file paths like `"files/code_changes.txt"` break when cells are re-run from different working directories.

**`merge_usage` reducer** — a 4-line custom function that enables cumulative token counting across parallel LLM calls. Without it, the second parallel agent's token counts would silently overwrite the first's.

**Three Pydantic output models:**

```
SimpleReviewOutput        — used by simple_review_node
  confidence: int
  findings:   str
  recommendations: str

FixItem                   — nested model for each required fix
  issue:       str        # what is wrong
  solution:    str        # how to fix it
  explanation: str        # why it matters

SummarizeFindingsOutput   — used by summarize_findings_node
  confidence:      int
  findings:        str
  fix:             List[FixItem]   ← typed list, not plain list
  recommendations: str
```

Using `List[FixItem]` instead of bare `list` means each fix item is **validated and typed** — the LLM must return structured objects with `issue`, `solution`, and `explanation` fields, not freeform strings. This makes the output programmatically usable (e.g. for generating GitHub review comments).

In [2]:
import os
import json
from pathlib import Path
from typing import TypedDict, Annotated, Optional, List
import operator

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END

# Load OPENAI_API_KEY (and any other vars) from .env in the project directory
load_dotenv(Path.cwd() / ".env")

# Resolve notebook directory once at import time — used to resolve relative PR file paths
NOTEBOOK_DIR = Path.cwd()

# Shared LLM — GPT-4o for all agents
llm = ChatOpenAI(model="gpt-4o", temperature=0)


# ------------------------------------------------------------------
# Reducer for token usage — sums numeric values across parallel nodes
# ------------------------------------------------------------------
def merge_usage(a: dict, b: dict) -> dict:
    return {k: a.get(k, 0) + b.get(k, 0) for k in set(a) | set(b)}


# ------------------------------------------------------------------
# Pydantic output models — structured responses from LLM agents
# ------------------------------------------------------------------
class SimpleReviewOutput(BaseModel):
    confidence:      int = Field(description="Confidence score 0-100 that the change is safe to merge")
    findings:        str = Field(description="Summary of key findings")
    recommendations: str = Field(description="Additional recommendations or observations")


class FixItem(BaseModel):
    """A single actionable fix item from the Tech Lead review."""
    issue:       str = Field(description="The specific problem that must be fixed")
    solution:    str = Field(description="The recommended fix or approach")
    explanation: str = Field(description="Why this must be fixed and the risk if not addressed")


class SummarizeFindingsOutput(BaseModel):
    confidence:      int        = Field(description="Confidence score 0-100 that the change is safe to merge")
    findings:        str        = Field(description="Summary of key findings from both quality and security analyses")
    fix:             List[FixItem] = Field(description="List of items the author MUST fix before merge, each with issue, solution, and explanation")
    recommendations: str        = Field(description="Additional recommendations or observations")


print("✅ Imports complete")

✅ Imports complete


## Step 2 — Define Shared State

`PRReviewState` is the **single source of truth** passed through every node. Understanding it is the key to understanding the whole graph.

### Field categories

```
┌─────────────────────────────────────────────────────────────┐
│  INPUT (set once by the caller before invoke)               │
│    pr_file_path  — path to the PR diff file                 │
├─────────────────────────────────────────────────────────────┤
│  DERIVED (written by one node, read by later nodes)         │
│    pr_content      — raw diff text            (read_pr)     │
│    review_type     — 'simple'/'full'/'error'  (router)      │
│    quality_findings — Senior Developer output  (full path)  │
│    security_findings — Security Engineer output (full path)  │
│    summary         — Tech Lead synthesis       (full path)  │
│    simple_review   — lightweight LLM output    (simple path)│
│    final_decision  — APPROVED / REQUEST CHANGES / REJECT    │
├─────────────────────────────────────────────────────────────┤
│  REDUCER FIELDS (written by multiple nodes, merged)         │
│    errors       Annotated[list, operator.add]               │
│    tokens_used  Annotated[dict, merge_usage]                │
│    messages     Annotated[list, operator.add]               │
└─────────────────────────────────────────────────────────────┘
```

### Why `Optional[str]` for path-specific fields?

`quality_findings` is only populated on the `full` path. On the `simple` path it stays `None`. `Optional[str]` makes the schema honest about this — downstream nodes use `.get()` with a fallback rather than assuming the field is always set.

### The `Annotated` reducer pattern

`errors: Annotated[list, operator.add]` tells LangGraph:

1. The type is `list`
2. When **multiple nodes** write to `errors`, **concatenate** the lists — don't overwrite

```python
# Node A returns: {"errors": ["file not found"]}
# Node B returns: {"errors": ["timeout"]}
# Final state:    {"errors": ["file not found", "timeout"]}  ← operator.add merges them
```

This is the same mechanism used for `messages` (audit log) and `tokens_used` (cost tracking).

In [3]:
class PRReviewState(TypedDict):
    """
    Shared state passed through the entire Code Review Tool graph.
    
    - pr_file_path    : path to the PR file (required — caller must supply)
    - pr_content      : raw diff text read from the PR file
    - review_type     : routing decision — 'simple', 'full', or 'error'
    - quality_findings: output from Senior Developer agent (full path only)
    - security_findings: output from Security Engineer agent (full path only)
    - summary         : Tech Lead's consolidated summary (full path only)
    - simple_review   : output from the simple LLM review path (simple path only)
    - final_decision  : the final accept/request-changes verdict
    - errors          : append-only list of errors encountered during execution
    - tokens_used     : cumulative token usage across all LLM calls (merged by summing)
    - messages        : append-only log of all agent messages (brief)
    """
    pr_file_path:      str
    pr_content:        str
    review_type:       str
    quality_findings:  Optional[str]
    security_findings: Optional[str]
    summary:           Optional[str]
    simple_review:     Optional[str]
    final_decision:    Optional[str]
    errors:            Annotated[list, operator.add]
    tokens_used:       Annotated[dict, merge_usage]
    messages:          Annotated[list, operator.add]

print("✅ State schema defined")

✅ State schema defined


## Step 3 — Read PR File (Graceful Error Handling)

### The error propagation pattern

Most frameworks raise exceptions when something goes wrong. In LangGraph, a better pattern is to **store errors in state** and let the router detect and route them:

```
Traditional approach:
  read_file() → raises FileNotFoundError → graph crashes with traceback

LangGraph pattern:
  read_file() → returns {"errors": ["PR file not found: ..."]}
                                 ↓
  route_review() → if state.get("errors"): return "error"
                                 ↓
                   return_final_answer() → prints error gracefully
```

### Why this is better

| Concern | Exception approach | State-error approach |
|---------|-------------------|---------------------|
| User experience | Ugly traceback | Clean error message |
| Auditability | Lost after crash | Stored in `state["errors"]` |
| Testability | Requires mocking exceptions | Assert on `result["errors"]` |
| Composability | One crash stops everything | Multiple errors can accumulate |

### Error as a first-class routing destination

`read_pr_file` returns `{"errors": [...], "messages": [...]}` with **no** `pr_content` key. The router in Step 4 sees the populated `errors` list and routes directly to `return_final_answer` via the `"error"` edge — completely bypassing all LLM calls and saving API costs.

In [4]:
def read_pr_file(state: PRReviewState) -> dict:
    """
    Reads the PR diff file from disk.
    On failure, stores the error in state — does NOT raise — so the router
    can detect errors and route to return_final_answer via the 'error' path.
    Maps to: 'Read PR file' node in Image 3 (Outer Router Graph).
    """
    path = Path(state["pr_file_path"])
    if not path.is_absolute():
        path = NOTEBOOK_DIR / path

    try:
        content = path.read_text()
        print(f"📄 Read PR file: {path.name} ({len(content)} chars)")
        return {
            "pr_content": content,
            "messages": [f"[read_pr] Loaded: {path.name} ({len(content)} chars)"]
        }
    except FileNotFoundError:
        msg = f"PR file not found: {path}"
        print(f"❌ {msg}")
        return {
            "errors": [msg],
            "messages": [f"[read_pr] ERROR: {msg}"]
        }
    except Exception as e:
        msg = f"Failed to read PR file: {e}"
        print(f"❌ {msg}")
        return {
            "errors": [msg],
            "messages": [f"[read_pr] ERROR: {msg}"]
        }

print("✅ read_pr_file tool defined")

✅ read_pr_file tool defined


## Step 4 — LLM-as-Router (Conditional Routing)

### The LLM-as-router pattern

Instead of hard-coded rules (`if "auth" in diff: full_review`), we delegate the routing decision to the LLM. This makes the router adaptive — adjusting routing criteria is as simple as editing the system prompt.

```
┌──────────────────────────────────────────────────────────────┐
│  System: "Respond with ONLY one word: simple or full"        │
│  Human:  "PR Diff:\n\n{pr_content}"                         │
└──────────────────────────────────────────────────────────────┘
              ↓
         decision = response.content.strip().lower()
              ↓
   ┌──────────┬──────────┬──────────────────┐
 "simple"  "full"    anything else
                           ↓
                   warn + default to "full"   ← safer default
```

### Two-function pattern: node + edge function

```python
def route_review_type(state) -> dict:   # NODE — calls LLM, writes review_type to state
    ...
    return {"review_type": "full", "tokens_used": {...}, ...}

def routing_edge(state) -> str:         # EDGE FUNCTION — reads state, returns node name
    return state["review_type"]         # "simple", "full", or "error"
```

LangGraph separates **computation** (nodes return state patches) from **routing** (edge functions return string labels). The edge function is called *after* the node has already updated state, so it always reads the freshest value.

### Error-first check

The router checks `state.get("errors")` *before* making the LLM call. If there's an upstream error (e.g. file not found), it returns `"error"` immediately without burning tokens.

### Safety defaulting

If the LLM returns unexpected output (e.g. `"complex"`), the router defaults to `"full"` — the more thorough path. Defaulting to `"simple"` would be dangerous because it could let risky code through a lightweight review.

In [5]:
def route_review_type(state: PRReviewState) -> dict:
    """
    LLM-driven router. First checks for upstream errors, then reads the PR
    diff and decides:
      - 'simple' : minor/cosmetic changes
      - 'full'   : logic changes, auth, security-sensitive, new functions
      - 'error'  : upstream error detected — skip review entirely
    
    Unexpected or failed LLM output defaults to 'full' (safe side).
    Maps to: 'Should Trigger AI Review?' node in Image 3.
    """
    # Propagate any upstream errors (e.g. file not found)
    if state.get("errors"):
        print(f"⚠️  Error detected upstream — skipping review")
        return {
            "review_type": "error",
            "messages": ["[router] Upstream error — routing to error handler"]
        }

    system_prompt = """You are a senior engineering manager triaging pull requests.
    
Given a code diff, classify it as either:
- 'simple'  : only minor/cosmetic changes (import aliases, whitespace, comments, variable renames with no logic change)
- 'full'    : contains logic changes, authentication code, security-sensitive operations, 
              new functions, database queries, or anything requiring deep review

Respond with ONLY one word: simple or full. No explanation."""

    try:
        response = llm.invoke([
            SystemMessage(content=system_prompt),
            HumanMessage(content=f"PR Diff:\n\n{state['pr_content']}")
        ])
        decision = response.content.strip().lower()
        usage = response.usage_metadata or {}
    except Exception as e:
        print(f"⚠️  Router LLM call failed ({e}) — defaulting to 'full' for safety")
        decision = "full"
        usage = {}

    if decision not in ("simple", "full"):
        print(f"⚠️  Unexpected router output: '{decision}' — defaulting to 'full' for safety")
        review_type = "full"
    else:
        review_type = decision

    print(f"🔀 Router decision: '{review_type}'")

    return {
        "review_type": review_type,
        "tokens_used": {"input_tokens": usage.get("input_tokens", 0),
                        "output_tokens": usage.get("output_tokens", 0),
                        "total_tokens": usage.get("total_tokens", 0)},
        "messages": [f"[router] Review type: {review_type}"]
    }

def routing_edge(state: PRReviewState) -> str:
    """Conditional edge — returns next node name based on review_type."""
    return state["review_type"]  # 'simple', 'full', or 'error'

print("✅ Router defined")

✅ Router defined


## Step 5 — Simple Review Node (Structured Output Pattern)

### The structured output + token capture pattern

This is the most reusable pattern in the notebook. Use it whenever you need **both** a validated typed response *and* token usage from the same LLM call:

```python
structured_llm = llm.with_structured_output(SimpleReviewOutput, include_raw=True)
raw_result = structured_llm.invoke([...])

# raw_result is a dict with two keys:
# {
#   "parsed": SimpleReviewOutput(confidence=95, findings="...", recommendations="..."),
#   "raw":    AIMessage(content="...", usage_metadata={"input_tokens": 300, ...})
# }

review_dict = raw_result["parsed"].model_dump()   # Pydantic → dict
usage       = raw_result["raw"].usage_metadata     # token counts from raw AIMessage
```

**Why `include_raw=True`?**
Without it, `with_structured_output` discards the raw `AIMessage` and returns only the parsed object — so `usage_metadata` is lost. `include_raw=True` preserves both.

### Why store as a JSON string in state?

The structured output is serialised with `json.dumps(review_dict)` before writing to `state["simple_review"]`. This keeps all `PRReviewState` fields as primitive strings — easy to persist to disk, log, and pass directly into the next LLM's prompt as context.

### Pydantic model for this path

```python
class SimpleReviewOutput(BaseModel):
    confidence:      int   # 0–100, how safe is this to merge?
    findings:        str   # what changed and is it correct?
    recommendations: str   # style notes, suggestions
```

In [6]:
def simple_review_node(state: PRReviewState) -> dict:
    """
    Lightweight LLM review for simple/cosmetic diffs.
    Returns structured output: {confidence, findings, recommendations}.
    Maps to: 'Simple Review' node in Image 3.
    """
    structured_llm = llm.with_structured_output(SimpleReviewOutput, include_raw=True)

    system_prompt = """You are a code reviewer. The change is minor and low-risk.
Review the PR diff and return a structured assessment with:
- confidence: integer 0–100 representing how confident you are the change is safe to merge
- findings: a concise summary of what changed and whether it is correct and safe
- recommendations: any style, convention notes, or suggestions for improvement"""

    try:
        raw_result = structured_llm.invoke([
            SystemMessage(content=system_prompt),
            HumanMessage(content=f"PR Diff:\n\n{state['pr_content']}")
        ])
        review_dict = raw_result["parsed"].model_dump()
        usage = raw_result["raw"].usage_metadata or {}
    except Exception as e:
        print(f"❌ Simple review LLM call failed: {e}")
        review_dict = {"confidence": 0, "findings": f"[ERROR] {e}", "recommendations": ""}
        usage = {}

    print(f"✏️  Simple review complete (confidence: {review_dict.get('confidence')})")

    return {
        "simple_review": json.dumps(review_dict),
        "tokens_used": {"input_tokens": usage.get("input_tokens", 0),
                        "output_tokens": usage.get("output_tokens", 0),
                        "total_tokens": usage.get("total_tokens", 0)},
        "messages": [f"[simple_review] complete (confidence: {review_dict.get('confidence')})"]
    }

print("✅ simple_review_node defined")

✅ simple_review_node defined


## Step 6 — Parallel Agents: Fan-Out Pattern

### What fan-out means

Fan-out is when a single node triggers *multiple* successor nodes that run **simultaneously** in separate threads. In LangGraph, achieve this with multiple `add_edge` calls from the same source:

```python
outer_builder.add_edge("full_review_start", "security_review")   # ─┐ run in parallel
outer_builder.add_edge("full_review_start", "quality_analysis")  # ─┘
```

Both agents receive the **same state snapshot** and execute concurrently.

### Why `full_review_start` exists

`route_review` already handles three paths (`simple`, `full`, `error`) via conditional edges — it can't also be a fan-out source. `full_review_start` is a **no-op fan-out trigger**: a dedicated node whose only purpose is to be the origin of two parallel edges.

```
route_review ──[full]──► full_review_start ──► security_review  (parallel)
                                           └──► quality_analysis (parallel)
```

### Token accumulation across parallel nodes

Both agents return a `tokens_used` dict. Because `tokens_used` uses the `merge_usage` reducer, LangGraph calls `merge_usage(existing, new)` for each update:

```
security_review  →  {input_tokens: 474, output_tokens: 497, total_tokens:  971}
quality_analysis →  {input_tokens: 439, output_tokens: 631, total_tokens: 1070}
                             ↓           merge_usage()           ↓
              final: {input_tokens: 913, output_tokens: 1128, total_tokens: 2041}
```

Without the custom reducer, the second write would silently overwrite the first and you'd lose half your token accounting.

| Agent | Role | Prompt focus |
|-------|------|-------------|
| `quality_analysis_node` | Senior Developer | Style, bugs, severity (CRITICAL/MAJOR/MINOR) |
| `security_review_node` | Security Engineer | Vulnerabilities, risk levels (HIGH/MEDIUM/LOW), BLOCKING verdict |

In [7]:
# ------------------------------------------------------------------
# AGENT 1: Senior Developer — Quality Analysis
# ------------------------------------------------------------------
def quality_analysis_node(state: PRReviewState) -> dict:
    system_prompt = """You are a Senior Developer performing a code quality review.

Analyse the PR diff and provide a structured report covering:

1. STYLE ISSUES
   - Naming conventions, formatting, code organisation
   - PEP8 / language-standard violations

2. POTENTIAL BUGS
   - Logic errors, edge cases not handled, incorrect assumptions
   - Resource leaks, exception handling gaps

3. SEVERITY CLASSIFICATION
   - List each issue as CRITICAL, MAJOR, or MINOR
   - Critical = must fix before merge. Minor = nice-to-have.

Be specific. Reference line numbers or code snippets where relevant."""

    try:
        response = llm.invoke([
            SystemMessage(content=system_prompt),
            HumanMessage(content=f"PR Diff to review:\n\n{state['pr_content']}")
        ])
        content = response.content
        usage = response.usage_metadata or {}
    except Exception as e:
        print(f"❌ Quality analysis LLM call failed: {e}")
        content = f"[ERROR] LLM call failed: {e}"
        usage = {}

    print("👨‍💻 Quality Analysis complete")

    return {
        "quality_findings": content,
        "tokens_used": {"input_tokens": usage.get("input_tokens", 0),
                        "output_tokens": usage.get("output_tokens", 0),
                        "total_tokens": usage.get("total_tokens", 0)},
        "messages": [f"[quality_analysis] complete ({len(content)} chars)"]
    }


# ------------------------------------------------------------------
# AGENT 2: Security Engineer — Security Review
# ------------------------------------------------------------------
def security_review_node(state: PRReviewState) -> dict:
    system_prompt = """You are a Security Engineer performing a security review of a code change.

Analyse the PR diff and provide a structured security report covering:

1. VULNERABILITIES FOUND
   - Injection attacks (SQL, command, XSS)
   - Authentication / authorisation flaws
   - Sensitive data exposure (logging PII, plaintext secrets)
   - Insecure cryptography or hashing
   - Timing attacks and race conditions

2. RISK LEVELS
   - Rate each finding as HIGH, MEDIUM, or LOW risk
   - Explain the attack vector briefly

3. BLOCKING ISSUES
   - List any issues that MUST be fixed before this PR can merge
   - Conclude with: BLOCKING or NON-BLOCKING

Be precise. Reference specific lines or patterns in the diff."""

    try:
        response = llm.invoke([
            SystemMessage(content=system_prompt),
            HumanMessage(content=f"PR Diff to review:\n\n{state['pr_content']}")
        ])
        content = response.content
        usage = response.usage_metadata or {}
    except Exception as e:
        print(f"❌ Security review LLM call failed: {e}")
        content = f"[ERROR] LLM call failed: {e}\n\nBLOCKING"
        usage = {}

    print("🔐 Security Review complete")

    return {
        "security_findings": content,
        "tokens_used": {"input_tokens": usage.get("input_tokens", 0),
                        "output_tokens": usage.get("output_tokens", 0),
                        "total_tokens": usage.get("total_tokens", 0)},
        "messages": [f"[security_review] complete ({len(content)} chars)"]
    }

print("✅ Quality Analysis and Security Review agents defined")

✅ Quality Analysis and Security Review agents defined


## Step 7 — Fan-In and Guardrail Pattern

### Fan-in: waiting for parallel agents

`aggregate_node` is the **fan-in join point**. LangGraph automatically waits for *all predecessor nodes* to finish before running `aggregate`. This is how the two parallel branches synchronise.

```
security_review  ──┐
                   ├──► aggregate   ← LangGraph holds here until BOTH complete
quality_analysis ──┘
```

`aggregate_node` itself is a no-op (`return {}`). Its value is purely structural — it gives LangGraph a single node to synchronise at.

### Guardrail pattern

A guardrail is a **conditional edge function** that enforces a safety check *without* making an LLM call. It inspects state and returns a routing label:

```python
def output_guardrail(state) -> str:           # edge function, not a node
    findings = state.get("security_findings", "")
    if "BLOCKING" in findings.upper() and "NON-BLOCKING" not in findings.upper():
        return "block"     # → blocked_review
    return "proceed"       # → summarize_findings
```

```
aggregate ──► output_guardrail(state) ──► "proceed" ──► summarize_findings
                                      └──► "block"   ──► blocked_review
```

**Why edge functions, not nodes?**
Edge functions run without creating a state update. Guardrail logic is pure routing — it doesn't *produce* anything, so there's nothing to write to state. Keeping it as an edge function avoids cluttering the state with intermediate flags.

### The `blocked_review` hard-stop pattern

When `output_guardrail` returns `"block"`, execution routes to `blocked_review_node`. This node writes a REJECT verdict into `state["summary"]` so `final_decision_node` can read it uniformly — regardless of whether the blocked path or the full path ran.

```
blocked_review writes: state["summary"] = "REJECT\n\nThis PR has been BLOCKED..."
final_decision reads:  state["summary"]   ← same field, same downstream code
```

### The `output_guardrail` — fires after synthesis

`summarize_findings` always runs first (on both BLOCKING and NON-BLOCKING paths), then `output_guardrail` inspects `security_findings` and routes to either `final_decision` (safe) or `blocked_review` (critical issues). This ensures every path through the graph produces structured fix items and recommendations.

In [8]:
def aggregate_node(state: PRReviewState) -> dict:
    """
    Fan-in join point — LangGraph waits for BOTH parallel agents (security_review
    and quality_analysis) to complete before this node runs.
    No-op: just signals readiness for the output guardrail.
    
    Maps to: implicit fan-in before 'Output guardrail' in Images 1 & 2.
    """
    print("🔗 Aggregate: both agents complete — running output guardrail")
    return {}


def output_guardrail(state: PRReviewState) -> str:
    """
    Output guardrail — runs after the aggregate (fan-in) node.
    Checks whether the security findings contain a BLOCKING verdict.
    
    Returns:
        'proceed'  → safe to continue to Tech Lead summarize_findings
        'block'    → hard stop, route to blocked_review
    
    Maps to: 'Output guardrail' in Images 1 & 2.
    """
    findings = state.get("security_findings", "")
    
    if "BLOCKING" in findings.upper() and "NON-BLOCKING" not in findings.upper():
        print("🚨 Output guardrail: BLOCKING issue detected — halting review")
        return "block"
    
    print("✅ Output guardrail: passed")
    return "proceed"


def blocked_review_node(state: PRReviewState) -> dict:
    """
    Hard-stop node reached when output_guardrail returns 'block'.
    Does NOT overwrite summary — structured JSON from summarize_findings_node
    is preserved so return_final_answer can show fix items and recommendations.
    
    Maps to: hard-stop path from 'Output guardrail' in Images 1 & 2.
    """
    print("🚫 Review BLOCKED — critical security issues must be resolved before merge")
    return {
        "messages": ["[blocked] Review halted by output guardrail — REJECT"]
    }


def report_guardrail(state: PRReviewState) -> str:
    """
    Report guardrail — runs after Summarize Findings.
    Validates the final summary is substantive before proceeding to END.
    
    Returns:
        'complete'  → summary is acceptable, proceed to END
    
    Maps to: 'Report guardrail' in Images 1 & 2.
    """
    summary = state.get("summary", "")
    
    if len(summary) < 100:
        print("⚠️  Report guardrail: summary suspiciously short — check Tech Lead output")
    else:
        print("✅ Report guardrail: passed")
    
    return "complete"

print("✅ Guardrails, aggregate_node, and blocked_review_node defined")

✅ Guardrails, aggregate_node, and blocked_review_node defined


## Step 8 — Tech Lead: Synthesis Node

### Combining multiple upstream fields

`summarize_findings_node` is the synthesis node — it reads *two* upstream agent outputs and combines them into a single structured verdict:

```
quality_findings  ──┐
                    ├──► LLM prompt ──► SummarizeFindingsOutput ──► state["summary"] (JSON)
security_findings ──┘
```

### The `fix` field as a machine-readable list

```python
class SummarizeFindingsOutput(BaseModel):
    confidence:      int   # 0–100, overall merge confidence
    findings:        str   # 2-3 sentence executive summary
    fix:             list  # items the author MUST fix — each with issue + solution
    recommendations: str   # nice-to-have improvements
```

Making `fix` a `list` (not a freeform string) keeps the output **machine-readable**. This structured data could drive:
- Automated GitHub PR comments (one comment per fix item)
- JIRA ticket creation
- AI-assisted fix generation (pass each `fix` item back to a coding agent)

### Reusing the `include_raw=True` pattern

Same as `simple_review_node` (Step 5) — `include_raw=True` preserves `usage_metadata` on the raw `AIMessage`. This is the **standard pattern** for any node that needs both structured data and token tracking:

```python
structured_llm = llm.with_structured_output(SummarizeFindingsOutput, include_raw=True)
raw_result     = structured_llm.invoke([...])
summary_dict   = raw_result["parsed"].model_dump()   # validated Pydantic → dict
usage          = raw_result["raw"].usage_metadata     # token counts
```

In [9]:
def summarize_findings_node(state: PRReviewState) -> dict:
    """
    Tech Lead agent — synthesises quality and security findings.
    Returns structured output: {confidence, findings, fix, recommendations}.
    Maps to: 'Summarize Findings' node in Image 2.
    """
    structured_llm = llm.with_structured_output(SummarizeFindingsOutput, include_raw=True)

    system_prompt = """You are the Tech Lead making the final code review decision.

You have received reports from two reviewers. Synthesise them into a structured review:
- confidence: integer 0–100 reflecting overall confidence that the PR is safe to merge as-is
- findings: a 2-3 sentence executive summary of what the PR does and the overall quality/security posture
- fix: a list of specific, actionable items the author MUST address before merge (each item: issue + solution + explanation)
- recommendations: any additional observations or nice-to-have improvements"""

    user_message = f"""QUALITY ANALYSIS (Senior Developer):
{state.get('quality_findings', 'No quality findings available.')}

---

SECURITY REVIEW (Security Engineer):
{state.get('security_findings', 'No security findings available.')}"""

    try:
        raw_result = structured_llm.invoke([
            SystemMessage(content=system_prompt),
            HumanMessage(content=user_message)
        ])
        summary_dict = raw_result["parsed"].model_dump()
        usage = raw_result["raw"].usage_metadata or {}
    except Exception as e:
        print(f"❌ Tech Lead LLM call failed: {e}")
        summary_dict = {"confidence": 0, "findings": f"[ERROR] {e}", "fix": [], "recommendations": ""}
        usage = {}

    print(f"👔 Tech Lead summary complete (confidence: {summary_dict.get('confidence')})")

    return {
        "summary": json.dumps(summary_dict),
        "tokens_used": {"input_tokens": usage.get("input_tokens", 0),
                        "output_tokens": usage.get("output_tokens", 0),
                        "total_tokens": usage.get("total_tokens", 0)},
        "messages": [f"[tech_lead_summary] complete (confidence: {summary_dict.get('confidence')})"]
    }

print("✅ summarize_findings_node defined")

✅ summarize_findings_node defined


## Step 9 — Final Decision Node & Terminal Node

### Path-agnostic final decision

`final_decision_node` handles all three convergence paths with a single implementation:

```python
if review_type == "simple":
    review_json = state.get("simple_review", "{}")   # from simple path
    label = "Simple Review"
else:
    review_json = state.get("summary", "{}")         # from full or blocked path
    label = "Full Code Review Tool"
```

It reads whichever field is populated, passes it to the LLM for verdict extraction, and writes `state["final_decision"]`.

### The terminal node pattern (`return_final_answer`)

`return_final_answer` is a **display-only terminal node** — it presents the answer but doesn't change state (`return {}`). Separating computation from presentation gives you:

- **Testability**: assert on `state["final_decision"]` without side effects
- **Single presentation point**: one place to add logging, webhooks, or notifications
- **Error path handling**: when the file wasn't found, `return_final_answer` is called directly from the router — there's no decision to make, just an error to display

### All paths converge

```
simple_review      ──┐
blocked_review     ──┼──► final_decision ──► return_final_answer ──► END
summarize_findings ──┘

route_review [error] ─────────────────────► return_final_answer ──► END
```

Every execution path ends at `return_final_answer`. The graph always reaches `END` cleanly — no dangling branches, no silent failures.

In [10]:
def final_decision_node(state: PRReviewState) -> dict:
    """
    Makes the final APPROVE / REQUEST CHANGES / REJECT decision via an LLM call.
    Works with whichever path ran (simple or full) by reading the structured
    review result stored in simple_review or summary.
    Maps to: 'Make final decision' in Image 3.
    """
    review_type = state.get("review_type", "unknown")

    if review_type == "simple":
        review_json = state.get("simple_review", "{}")
        label = "Simple Review"
    else:
        review_json = state.get("summary", "{}")
        label = "Full Code Review Tool"

    system_prompt = """You are a tech lead making the final merge decision on a pull request.

Given the review results below, output exactly one of:
  APPROVED          — safe to merge as-is
  REQUEST CHANGES   — mergeable after the listed fixes are applied
  REJECT            — critical issues; must not merge

Follow with 1-2 sentences explaining your reasoning."""

    try:
        response = llm.invoke([
            SystemMessage(content=system_prompt),
            HumanMessage(content=f"Review result ({label}):\n{review_json}")
        ])
        decision_text = response.content
        usage = response.usage_metadata or {}
    except Exception as e:
        print(f"❌ Final decision LLM call failed: {e}")
        decision_text = f"REJECT\n\n[ERROR] Could not make final decision: {e}"
        usage = {}

    verdict = "UNKNOWN"
    for keyword in ["APPROVED", "REQUEST CHANGES", "REJECT"]:
        if keyword in decision_text.upper():
            verdict = keyword
            break

    print(f"\n{'='*60}")
    print(f"🏁 FINAL DECISION ({label}): {verdict}")
    print(f"{'='*60}\n")

    return {
        "final_decision": f"[{label}] {decision_text}",
        "tokens_used": {"input_tokens": usage.get("input_tokens", 0),
                        "output_tokens": usage.get("output_tokens", 0),
                        "total_tokens": usage.get("total_tokens", 0)},
        "messages": [f"[final_decision] {verdict}"]
    }


def return_final_answer(state: PRReviewState) -> dict:
    """
    Terminal node — prints a detailed final report for developers.
    Always surfaces structured fix items, recommendations, confidence,
    and security findings (when blocking) across ALL flow paths.
    """
    if state.get("errors"):
        print("❌ Flow ended with errors:")
        for err in state["errors"]:
            print(f"   • {err}")

    answer = state.get("final_decision") or "No decision reached."
    print("\n" + "=" * 60)
    print("FINAL ANSWER")
    print("=" * 60)
    print(answer)

    review_type = state.get("review_type", "")
    summary_str = state.get("summary") or ""

    # Full review — always show structured fix items and recommendations
    if review_type == "full" and summary_str:
        try:
            parsed = SummarizeFindingsOutput.model_validate_json(summary_str)

            if parsed.fix:
                print("\n" + "─" * 60)
                print(f"REQUIRED FIXES ({len(parsed.fix)} item{'s' if len(parsed.fix) != 1 else ''} — must address before merge):")
                print("─" * 60)
                for i, item in enumerate(parsed.fix, 1):
                    print(f"\n  {i}. Issue:      {item.issue}")
                    print(f"     Solution:   {item.solution}")
                    print(f"     Why:        {item.explanation}")

            if parsed.recommendations:
                print("\n" + "─" * 60)
                print("RECOMMENDATIONS (nice-to-have):")
                print("─" * 60)
                print(f"  {parsed.recommendations}")

            print("\n" + "─" * 60)
            print(f"MERGE CONFIDENCE: {parsed.confidence}/100")

        except Exception as e:
            print(f"⚠️  Could not parse structured summary: {e}")

    # Always show security findings when blocking issues were found
    security = state.get("security_findings") or ""
    if review_type == "full" and "BLOCKING" in security.upper():
        print("\n" + "─" * 60)
        print("BLOCKING SECURITY FINDINGS:")
        print("─" * 60)
        print(security)

    # Quality judge score (v2 graph only)
    judge_score  = state.get("judge_score")
    judge_reason = state.get("judge_reason")
    if judge_score is not None:
        print("\n" + "─" * 60)
        print(f"QUALITY JUDGE: {judge_score}/10 — {judge_reason or ''}")

    print("\n" + "=" * 60)
    return {}
print("✅ final_decision_node and return_final_answer defined")

✅ final_decision_node and return_final_answer defined


## Step 10 — Subgraph Visualisation: Parallel Code Review Tool

This cell builds the parallel review subgraph **in isolation** — useful for visualisation and incremental development, but intentionally **not embedded** in the outer router graph.

### Why not embed the compiled subgraph?

```
❌ Problematic approach:
   outer_graph.add_node("parallel", parallel_code_review)  ← compiled subgraph as node

   Bug: The outer graph passes its full state (including the existing `messages` list)
        into the subgraph. The subgraph accumulates messages internally, then returns
        the full accumulated list. The outer graph's `operator.add` reducer appends
        this returned list onto the existing list → every message appears twice.

✅ Correct approach (used in Step 11):
   Inline all subgraph nodes directly in the outer graph.
   Each node appends only its own message — no duplication.
```

> **Rule of thumb:** When a subgraph shares `Annotated` reducer fields with its parent, inline the nodes. Only use compiled subgraphs as embedded nodes when state fields are fully isolated between parent and child.

### Standalone subgraph flow

```
START ──┬──► security_review  ──┐
        └──► quality_analysis ──┴──► aggregate ──[output_guardrail]──┬──► summarize_findings ──[output_guardrail]──┬──► final_decision ──► END
                                                                      └──► blocked_review ─────────────────────────► END
```

> **Learning tip:** Build complex graphs incrementally. Define the parallel subgraph first, visualise it, test it standalone, then inline its nodes into the full graph.

In [11]:
# ------------------------------------------------------------------
# Build the parallel subgraph (kept for standalone visualization in Cell 13)
# NOTE: this compiled graph is NOT used as a node in the outer router graph.
# The outer graph (Cell 12) inlines these nodes directly to avoid the
# subgraph state-merge issue that causes message duplication with operator.add.
# ------------------------------------------------------------------
parallel_builder = StateGraph(PRReviewState)

parallel_builder.add_node("security_review",   security_review_node)
parallel_builder.add_node("quality_analysis",  quality_analysis_node)
parallel_builder.add_node("aggregate",          aggregate_node)
parallel_builder.add_node("blocked_review",     blocked_review_node)
parallel_builder.add_node("summarize_findings", summarize_findings_node)

parallel_builder.add_edge(START, "security_review")
parallel_builder.add_edge(START, "quality_analysis")
parallel_builder.add_edge("security_review",  "aggregate")
parallel_builder.add_edge("quality_analysis", "aggregate")
# Always synthesise, then check guardrail
parallel_builder.add_edge("aggregate", "summarize_findings")
parallel_builder.add_conditional_edges(
    "summarize_findings",
    output_guardrail,
    {"proceed": END, "block": "blocked_review"}
)
parallel_builder.add_edge("blocked_review", END)

parallel_code_review = parallel_builder.compile()

print("✅ Parallel Code Review Tool (subgraph) compiled — used for visualization only")
print("\nSubgraph structure:")
try:
    parallel_code_review.get_graph().print_ascii()
except ImportError:
    print("(grandalf not installed — run Cell 1 to install, then re-run this cell)")

✅ Parallel Code Review Tool (subgraph) compiled — used for visualization only

Subgraph structure:
                   +-----------+                    
                   | __start__ |                    
                   +-----------+                    
                 ***            ***                 
               **                  **               
             **                      **             
+------------------+           +-----------------+  
| quality_analysis |           | security_review |  
+------------------+           +-----------------+  
                 ***            ***                 
                    **        **                    
                      **    **                      
                   +-----------+                    
                   | aggregate |                    
                   +-----------+                    
                         *                          
                         *                          


## Step 11 — Build the Complete Router Graph

All nodes are inlined in a single `StateGraph`. This section shows **how** the complete graph is assembled and **why** each wiring decision was made.

### Graph assembly recipe

```python
# 1. Create the builder
outer_builder = StateGraph(PRReviewState)

# 2. Register all nodes
outer_builder.add_node("read_pr", read_pr_file)
# ... (all 11 nodes)

# 3. Wire the linear start
outer_builder.add_edge(START, "read_pr")
outer_builder.add_edge("read_pr", "route_review")

# 4. Three-way conditional router
outer_builder.add_conditional_edges(
    "route_review", routing_edge,
    {"simple": "simple_review", "full": "full_review_start", "error": "return_final_answer"}
)

# 5. Fan-out from full_review_start
outer_builder.add_edge("full_review_start", "security_review")   # ─┐ parallel
outer_builder.add_edge("full_review_start", "quality_analysis")  # ─┘

# 6. Fan-in to aggregate
outer_builder.add_edge("security_review",  "aggregate")
outer_builder.add_edge("quality_analysis", "aggregate")

# 7. Always synthesise first — gives recommendations on ALL paths
outer_builder.add_edge("aggregate", "summarize_findings")
outer_builder.add_conditional_edges("summarize_findings", output_guardrail,
    {"proceed": "final_decision", "block": "blocked_review"})
    {"proceed": "summarize_findings", "block": "blocked_review"})

# 8. All full-review outcomes → final_decision
outer_builder.add_edge("blocked_review",    "final_decision")
    {"complete": "final_decision"})

# 9. Convergence
outer_builder.add_edge("simple_review",        "final_decision")
outer_builder.add_edge("final_decision",        "return_final_answer")
outer_builder.add_edge("return_final_answer",  END)

# 10. Compile
code_review_app = outer_builder.compile()
```

### Complete flow at a glance (v1 — base graph)

```
START ──► read_pr ──► route_review
                           │
             ┌─────────────┼──────────────┐
          [simple]       [full]        [error]
             │             │               │
       simple_review  full_review_start    │
             │          ┌──┴──┐            │
             │  security_rev  quality_ana  │
             │          └──┬──┘            │
             │          aggregate          │
             │       [output_guardrail]    │
             │       ┌──────┴──────┐       │
             │   [block]       [proceed]   │
             │  blocked_rev  summarize_fin │
             │       └──────┬──────┘       │
             └──────► final_decision ◄─────┘
                            │
                    return_final_answer
                            │
                           END
```

> **Step 23 extends this graph** by inserting a `quality_judge` node between `quality_analysis` and `aggregate`. `quality_judge` uses `gpt-4o-mini` to score the analysis and can loop `quality_analysis` back with a stricter prompt (up to 2 retries) before proceeding to `aggregate`. The extended graph is built as `code_review_app_v2`.

In [12]:
def full_review_start(state: PRReviewState) -> dict:
    """Fan-out entry for the full-review path — starts both parallel agents."""
    print("⚡ Full review: launching Security Review + Quality Analysis in parallel")
    return {}


# ------------------------------------------------------------------
# Build the complete graph with all nodes inlined (no embedded subgraph)
# ------------------------------------------------------------------
outer_builder = StateGraph(PRReviewState)

# Nodes
outer_builder.add_node("read_pr",              read_pr_file)
outer_builder.add_node("route_review",         route_review_type)
outer_builder.add_node("simple_review",        simple_review_node)
outer_builder.add_node("full_review_start",    full_review_start)
outer_builder.add_node("security_review",      security_review_node)
outer_builder.add_node("quality_analysis",     quality_analysis_node)
outer_builder.add_node("aggregate",            aggregate_node)
outer_builder.add_node("blocked_review",       blocked_review_node)
outer_builder.add_node("summarize_findings",   summarize_findings_node)
outer_builder.add_node("final_decision",       final_decision_node)
outer_builder.add_node("return_final_answer",  return_final_answer)

# Linear start
outer_builder.add_edge(START,       "read_pr")
outer_builder.add_edge("read_pr",   "route_review")

# Router: simple → simple_review | full → full_review_start | error → return_final_answer
outer_builder.add_conditional_edges(
    "route_review",
    routing_edge,
    {"simple": "simple_review", "full": "full_review_start", "error": "return_final_answer"}
)

# Fan-out → parallel agents
outer_builder.add_edge("full_review_start", "security_review")
outer_builder.add_edge("full_review_start", "quality_analysis")

# Fan-in → aggregate
outer_builder.add_edge("security_review",  "aggregate")
outer_builder.add_edge("quality_analysis", "aggregate")

# Always synthesise first — gives recommendations on ALL paths
outer_builder.add_edge("aggregate", "summarize_findings")

# Output guardrail fires after synthesis
outer_builder.add_conditional_edges(
    "summarize_findings",
    output_guardrail,
    {"proceed": "final_decision", "block": "blocked_review"}
)

# Both full-review outcomes → final_decision
outer_builder.add_edge("blocked_review", "final_decision")

# Simple path → final_decision
outer_builder.add_edge("simple_review",  "final_decision")

# All paths converge at return_final_answer → END
outer_builder.add_edge("final_decision",      "return_final_answer")
outer_builder.add_edge("return_final_answer", END)

# Compile
code_review_app = outer_builder.compile()

print("✅ Code Review App compiled")
print("\nFull graph structure:")
try:
    code_review_app.get_graph().print_ascii()
except ImportError:
    print("(grandalf not installed — run Cell 1 to install, then re-run this cell)")

✅ Code Review App compiled

Full graph structure:
                                                 +-----------+                                        
                                                 | __start__ |                                        
                                                 +-----------+                                        
                                                       *                                              
                                                       *                                              
                                                       *                                              
                                                  +---------+                                         
                                                  | read_pr |                                         
                                                  +---------+                                         
                       

## Next: Notebook 2 — Demo & Patterns

The graph is now compiled as `code_review_app`. Open **`02_demo_and_patterns.ipynb`** to:
- Run all 4 execution paths (REJECT, REQUEST CHANGES, APPROVED, error)
- See real-time streaming output
- Inspect the audit trail and token costs
- Explore state persistence with SqliteSaver